# 02 · Train the pix2pix generator

Uses the tiles from `01_data.ipynb` (on Google Drive). Runtime: **GPU (T4)**.

1. Sanity check: the model must be able to memorise 20 tiles. If it cannot, there is a bug
   and a full run would waste GPU time.
2. Full training run. Checkpoints go to Drive after every epoch, so if Colab disconnects,
   just re-run this notebook: it resumes from the last epoch.
3. Loss curves.

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/muhammedshahidsds25-collab/disaster-gen.git"

drive.mount("/content/drive")
!git clone -q {REPO_URL} /content/disaster-gen || git -C /content/disaster-gen pull -q
%cd /content/disaster-gen
# Regular (not editable) install: an editable src-layout install is not importable
# until the runtime restarts.
!pip install -q .

In [ ]:
import dataclasses
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from disastergen.config import load_config
from disastergen.data.dataset import TileDataset, compute_stats
from disastergen.models import build_models
from disastergen.train import fit, make_optimizers, set_seed, train_step

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
TILE_DIR = DRIVE_ROOT / "tiles"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
OUT_DIR = DRIVE_ROOT / "outputs" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

## 1. Overfit sanity check (20 most severe tiles, 200 steps)

In [ ]:
set_seed(cfg.train.seed)
all_tiles = sorted(TILE_DIR.glob("*.npz"))
probe = TileDataset(all_tiles, compute_stats(all_tiles), augment=False)
top20 = [all_tiles[i] for i in probe.severities.argsort()[-20:]]
loader = DataLoader(TileDataset(top20, probe.stats, augment=False), batch_size=4, shuffle=True)

g, d = (m.to(device) for m in build_models(cfg.model))
opt_g, opt_d = make_optimizers(g, d, cfg.train)
l1_curve = []
while len(l1_curve) < 200:
    for x, y in loader:
        losses = train_step(g, d, opt_g, opt_d, x.to(device), y.to(device), cfg.train.l1_weight)
        l1_curve.append(losses["g_l1"])
print(f"L1 first 10 steps: {sum(l1_curve[:10]) / 10:.3f}  last 10: {sum(l1_curve[-10:]) / 10:.3f}")
plt.plot(l1_curve)
plt.xlabel("step")
plt.ylabel("L1")
plt.title("Overfit check: L1 must drop clearly")
del g, d, opt_g, opt_d

## 2. Full training run (resumes automatically)

In [ ]:
run_cfg = dataclasses.replace(
    cfg, train=dataclasses.replace(cfg.train, checkpoint_dir=str(CKPT_DIR))
)
best_generator = fit(run_cfg, TILE_DIR, resume=True)
best_generator

## 3. Loss curves

In [ ]:
history = pd.read_csv(CKPT_DIR / "history.csv")
fig, (ax_gan, ax_l1) = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x="epoch", y=["d_loss", "g_gan"], ax=ax_gan, title="Adversarial losses")
history.plot(x="epoch", y=["g_l1", "val_l1"], ax=ax_l1, title="L1: train vs validation")
fig.tight_layout()
fig.savefig(OUT_DIR / "losses.png", dpi=150)
history.tail()